In [1]:
import pandas as pd
import numpy as np
import os
import json

def create_feature_engineering_summary():
    """Create a comprehensive summary of feature engineering results"""
    # Load data from previous notebooks
    
    # 1. Load transformation details
    transformation_details = {}
    try:
        with open('../../src/models/transformation_details.json', 'r') as f:
            transformation_details = json.load(f)
        print(f"Loaded transformation details for {len(transformation_details)} features")
    except Exception as e:
        print(f"Could not load transformation details: {e}")
    
    # 2. Load feature info (from composite_features)
    feature_info = {}
    try:
        with open('../../src/models/feature_info.json', 'r') as f:
            feature_info = json.load(f)
        print(f"Loaded feature info with {len(feature_info.get('original_features', []))} original features")
    except Exception as e:
        print(f"Could not load feature info: {e}")
    
    # 3. Load selected features
    selected_features = []
    try:
        with open('../../src/models/selected_features.txt', 'r') as f:
            selected_features = [line.strip() for line in f if line.strip()]
        print(f"Loaded {len(selected_features)} selected features")
    except Exception as e:
        print(f"Could not load selected features: {e}")
    
    # 4. Load stable features
    stable_features = []
    try:
        with open('../../src/models/stable_features.txt', 'r') as f:
            stable_features = [line.strip() for line in f if line.strip()]
        print(f"Loaded {len(stable_features)} stable features")
    except Exception as e:
        print(f"Could not load stable features: {e}")
    
    # 5. Load feature stability metrics
    stability_metrics = {}
    try:
        with open('../../src/models/feature_stability.json', 'r') as f:
            stability_metrics = json.load(f)
        print(f"Loaded stability metrics for {len(stability_metrics)} features")
    except Exception as e:
        print(f"Could not load stability metrics: {e}")
    
    # 6. Load attack-specific features
    attack_features = {}
    attack_types = ["brute_force-web", "sql_injection", "brute_force-xss", "general"]
    for attack_type in attack_types:
        try:
            filename = f"../../src/models/{attack_type}_features.txt"
            if os.path.exists(filename):
                with open(filename, 'r') as f:
                    features = [line.strip() for line in f if line.strip()]
                attack_features[attack_type] = {
                    'top_features': features,
                    'all_importances': [(f, 1.0) for f in features]  # Dummy importance values
                }
                print(f"Loaded {len(features)} features for {attack_type}")
        except Exception as e:
            print(f"Could not load features for {attack_type}: {e}")
    
    # Start building the summary
    summary = """
# Feature Engineering Results Summary

## Transformation Summary

| Transformation Type | Count | Examples |
|-------------|-------|----------|
"""
    
    # Count transformation types
    transform_counts = {}
    for feature, details in transformation_details.items():
        transform_type = details.get("type", "none")
        transform_counts[transform_type] = transform_counts.get(transform_type, 0) + 1
    
    # Add to summary
    for transform_type, count in transform_counts.items():
        examples = []
        for feature, details in transformation_details.items():
            if details.get("type") == transform_type:
                examples.append(feature)
                if len(examples) >= 3:
                    break
        examples_str = ", ".join(examples)
        summary += f"| {transform_type} | {count} | {examples_str} |\n"
    
    # Add new features section
    summary += """
## New Features Created

| Feature Type | Count | Features |
|-------------|-------|----------|
"""
    
    # New feature types from feature_info
    ratio_features = feature_info.get('ratio_features', [])
    attack_specific_features = feature_info.get('attack_specific_features', [])
    
    # Categorize new features
    new_feature_categories = {
        'Ratio': [f for f in ratio_features if '_Ratio' in f or 'Ratio' in f],
        'Attack Score': [f for f in attack_specific_features if '_Score' in f],
        'Flag Combination': [f for f in ratio_features if 'Flag_Combination' in f or 'Combination' in f],
        'Rate': [f for f in ratio_features if 'Per_Second' in f or '_Per_' in f]
    }
    
    # Add to summary
    for feature_type, features in new_feature_categories.items():
        if features:
            examples_str = ", ".join(features)
            summary += f"| {feature_type} | {len(features)} | {examples_str} |\n"
    
    # Add attack-specific section if we have attack features
    if attack_features:
        summary += """
## Attack-Specific Feature Importance

| Attack Type | Top Features |
|-------------|----------------|
"""
        
        # Add attack-specific features
        for attack_type, result in attack_features.items():
            top_features = result['top_features'][:5]  # Limit to top 5
            top_str = ", ".join(top_features)
            pretty_name = attack_type.replace('_', ' ').replace('-', ' ').title()
            summary += f"| {pretty_name} | {top_str} |\n"
    
    # Add final feature count
    original_count = len(feature_info.get('original_features', []))
    ratio_count = len(feature_info.get('ratio_features', []))
    attack_specific_count = len(feature_info.get('attack_specific_features', []))
    
    summary += f"""
## Final Feature Selection

- Initial feature count: {original_count}
- New ratio features created: {ratio_count}
- New attack-specific features created: {attack_specific_count}
- Final selected features: {len(selected_features)}
- Stable important features: {len(stable_features)}

## Key Findings

1. The most important features for detecting attacks are: {', '.join(stable_features[:5]) if stable_features else 'N/A'}
2. Attack-specific scores (like SQL_Injection_Score, Brute_Force_Score) significantly improve detection capability
3. Feature transformations have normalized the distributions and improved model input quality
4. {len(feature_info.get('original_features', [])) + len(feature_info.get('ratio_features', [])) + len(feature_info.get('attack_specific_features', [])) - len(selected_features)} features were removed due to high correlation or low importance

## Next Steps

1. Use these engineered features for model training
2. Implement the preprocessing pipeline in the API
3. Update the model to utilize the new features
"""
    
    return summary

# Create summary
feature_engineering_summary = create_feature_engineering_summary()
   
# Write summary to file
try:
    os.makedirs('../../docs', exist_ok=True)
    with open('../../docs/feature_engineering_summary.md', 'w') as f:
        f.write(feature_engineering_summary)
    print("Feature engineering summary saved to docs/feature_engineering_summary.md")
except Exception as e:
    print(f"Error saving summary: {e}")
    print("\nSummary content:")
    print(feature_engineering_summary)

Loaded transformation details for 77 features
Loaded feature info with 77 original features
Loaded 11 selected features
Loaded 7 stable features
Loaded stability metrics for 11 features
Loaded 15 features for brute_force-web
Feature engineering summary saved to docs/feature_engineering_summary.md
